# 04 - GraphRAG Retrievers

Progress from simple vector search to graph-enhanced retrieval that uses the aircraft topology.

**Prerequisites:** run **03_data_and_embeddings** first (creates the chunks, embeddings, indexes, and cross-links).

## What you'll learn
- `VectorRetriever` for semantic search, wrapped in a `GraphRAG` answer pipeline
- `VectorCypherRetriever` for graph-enhanced retrieval:
  - adjacent chunks for complete procedures
  - traversal to aircraft systems and components
  - traversal to extracted operating limits

## Section 1: Configuration

In [ ]:
%pip install neo4j-graphrag mlflow

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

NEO4J_URI = ""
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: enter your Neo4j credentials above before running!")
else:
    print(f"Configuration ready - {NEO4J_URI}")

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG

from data_utils import Neo4jConnection, get_llm, get_embedder

neo4j = Neo4jConnection(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD).verify()
driver = neo4j.driver

llm = get_llm()
embedder = get_embedder()
INDEX_NAME = "maintenanceChunkEmbeddings"
print(f"LLM: {llm.model_id} | Embedder: {embedder.model_id}")

---

# Part 1: Vector Retriever

Semantic search over the maintenance chunks, then a `GraphRAG` pipeline that feeds the retrieved context to the LLM.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver, index_name=INDEX_NAME, embedder=embedder, return_properties=["text"],
)

query = "What are the steps to troubleshoot engine vibration?"
result = vector_retriever.search(query_text=query, top_k=5)
print(f'Query: "{query}"  ({len(result.items)} results)\n' + "=" * 70)
for item in result.items:
    print(f"\nscore={item.metadata['score']:.4f}")
    print(f"  {item.content[:200]}...")

In [ ]:
query = "What are the normal EGT operating limits for the V2500 engine at different power settings?"

rag = GraphRAG(llm=llm, retriever=vector_retriever)
response = rag.search(
    query, retriever_config={"top_k": 5}, return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)
print(f'Query: "{query}"  ({len(response.retriever_result.items)} chunks)\n' + "=" * 70)
print("\nAnswer:\n" + response.answer)

---

# Part 2: Vector Cypher Retriever

`VectorCypherRetriever` uses vector search as the entry point, then runs a custom Cypher query to gather richer context from the graph.

## Example 1: Adjacent chunks

Procedures often span multiple chunks. Follow `NEXT_CHUNK` backward and forward to include surrounding context.

In [ ]:
adjacent_chunks_query = """
WITH node
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
RETURN doc.documentId AS document_id,
       node.index AS chunk_index,
       COALESCE(prev.text, '') AS previous_context,
       node.text AS main_context,
       COALESCE(next.text, '') AS next_context
"""

adjacent_retriever = VectorCypherRetriever(
    driver=driver, index_name=INDEX_NAME, embedder=embedder,
    retrieval_query=adjacent_chunks_query,
)

query = "How do I perform the engine vibration diagnostic flow?"
rag = GraphRAG(llm=llm, retriever=adjacent_retriever)
response = rag.search(
    query, retriever_config={"top_k": 3}, return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)
print(f'Query: "{query}"\n' + "=" * 70)
print("\nAnswer:\n" + response.answer)

## Example 2: Connecting to the aircraft topology

Start from a matched chunk, traverse `Document -[:APPLIES_TO]-> Aircraft` (created in notebook 03), then walk into systems and components. The retriever returns structured topology alongside the text.

In [ ]:
system_context_query = """
WITH node
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)-[:APPLIES_TO]->(a:Aircraft)
MATCH (a)-[:HAS_SYSTEM]->(s:System)
OPTIONAL MATCH (s)-[:HAS_COMPONENT]->(comp:Component)
WITH node, doc, a, s, comp
RETURN doc.aircraftType AS aircraft_type,
       a.tail_number AS aircraft,
       COLLECT(DISTINCT s.name)[0..3] AS systems,
       COLLECT(DISTINCT comp.name)[0..3] AS components,
       node.text AS context
"""

system_retriever = VectorCypherRetriever(
    driver=driver, index_name=INDEX_NAME, embedder=embedder,
    retrieval_query=system_context_query,
)

query = "What maintenance is required for the engine fuel pump?"
rag = GraphRAG(llm=llm, retriever=system_retriever)
response = rag.search(
    query, retriever_config={"top_k": 3}, return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)
print(f'Query: "{query}"\n' + "=" * 70)
print("\nAnswer:\n" + response.answer)
print("\nContext with system connections:")
for item in response.retriever_result.items:
    print(f"\n{item.content}")

## Example 3: Operating limits from extracted entities

Traverse the full chain `Chunk -> Document -> Aircraft -> System -> Sensor -> OperatingLimit`. The retriever returns both the text and the structured limits the LLM extracted in notebook 03, so the answer can cite specific numeric thresholds.

In [ ]:
operating_limit_query = """
WITH node
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)-[:APPLIES_TO]->(a:Aircraft)
OPTIONAL MATCH (a)-[:HAS_SYSTEM]->(:System)-[:HAS_SENSOR]->(s:Sensor)-[:HAS_LIMIT]->(ol:OperatingLimit)
WITH node, doc,
     COLLECT(DISTINCT {
         sensor: s.type, parameter: ol.parameterName,
         max: ol.maxValue, unit: ol.unit, regime: ol.regime
     })[0..5] AS operating_limits
RETURN doc.aircraftType AS aircraft_type, operating_limits, node.text AS context
"""

limit_retriever = VectorCypherRetriever(
    driver=driver, index_name=INDEX_NAME, embedder=embedder,
    retrieval_query=operating_limit_query,
)

query = "What are the EGT temperature limits for the engine?"
rag = GraphRAG(llm=llm, retriever=limit_retriever)
response = rag.search(
    query, retriever_config={"top_k": 3}, return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)
print(f'Query: "{query}"\n' + "=" * 70)
print("\nAnswer:\n" + response.answer)
print("\nContext with operating limits:")
for item in response.retriever_result.items:
    print(f"\n{item.content}")

---

# Part 3: Comparing strategies

Same query, two retrievers: plain vector text versus adjacent-chunk graph context.

In [ ]:
comparison_query = "What should I do if the EGT exceeds normal limits?"
print(f'Query: "{comparison_query}"\n' + "=" * 70)

print("\n[1] VECTOR RETRIEVER (text only)\n" + "-" * 40)
print(GraphRAG(llm=llm, retriever=vector_retriever).search(
    comparison_query, retriever_config={"top_k": 3},
    response_fallback="No relevant maintenance procedures found.",
).answer)

print("\n" + "=" * 70)
print("\n[2] VECTOR CYPHER RETRIEVER (adjacent chunks)\n" + "-" * 40)
print(GraphRAG(llm=llm, retriever=adjacent_retriever).search(
    comparison_query, retriever_config={"top_k": 3},
    response_fallback="No relevant maintenance procedures found.",
).answer)

| Retriever | Context | Graph traversal | Best for |
|-----------|---------|-----------------|----------|
| `VectorRetriever` | Raw text chunks | None | Quick lookups |
| `VectorCypherRetriever` (adjacent) | Text + surrounding chunks | `NEXT_CHUNK` | Procedures |
| `VectorCypherRetriever` (topology) | Text + systems/components | `APPLIES_TO`, `HAS_SYSTEM` | Aircraft-specific questions |
| `VectorCypherRetriever` (limits) | Text + operating limits | Full chain to `OperatingLimit` | Threshold queries |

**Next:** **05_hybrid_retrievers** adds keyword precision for technical terms like fault codes and part numbers.

In [ ]:
neo4j.close()